<a href="https://colab.research.google.com/github/prashanth-ds-ml/IEEE_CIS_Fraud_detection/blob/main/IEEE_CIS_Fraud_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IEEE-CIS Fraud Detection (Correct & Interview-Safe Notebook)

## Goal
Build a fraud classification model using **clean ML hygiene**:
- Split data first (avoid leakage)
- Use proper preprocessing
- Evaluate with ROC-AUC + confusion matrix
- Tune threshold to reduce false positives (risk-aware)

## Why this notebook is "trainer-grade"
This notebook is structured to teach:
- Why leakage happens in fraud datasets
- Why accuracy is misleading in imbalance
- How to build a reproducible pipeline using sklearn


In [1]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix, classification_report

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier

import warnings
warnings.filterwarnings("ignore")

RANDOM_STATE = 42

def print_basic_info(df, name="df"):
    print(f"--- {name} ---")
    print("Shape:", df.shape)
    print("Columns:", len(df.columns))
    print(df.head(2))


## Dataset Files (Expected in `/content/`)
Upload these Kaggle files to Colab (or mount Drive):
- `train_transaction.csv`
- `train_identity.csv`

Optional (not needed for training today):
- `test_transaction.csv`
- `test_identity.csv`


In [2]:
DATA_DIR = "/content"

train_trans_path = os.path.join(DATA_DIR, "train_transaction.csv")
train_id_path    = os.path.join(DATA_DIR, "train_identity.csv")

train_trans = pd.read_csv(train_trans_path)
train_id    = pd.read_csv(train_id_path)

print_basic_info(train_trans, "train_transaction")
print_basic_info(train_id, "train_identity")



--- train_transaction ---
Shape: (590540, 394)
Columns: 394
   TransactionID  isFraud  TransactionDT  TransactionAmt ProductCD  card1  \
0        2987000        0          86400            68.5         W  13926   
1        2987001        0          86401            29.0         W   2755   

   card2  card3       card4  card5  ... V330  V331  V332  V333  V334 V335  \
0    NaN  150.0    discover  142.0  ...  NaN   NaN   NaN   NaN   NaN  NaN   
1  404.0  150.0  mastercard  102.0  ...  NaN   NaN   NaN   NaN   NaN  NaN   

  V336  V337  V338  V339  
0  NaN   NaN   NaN   NaN  
1  NaN   NaN   NaN   NaN  

[2 rows x 394 columns]
--- train_identity ---
Shape: (144233, 41)
Columns: 41
   TransactionID  id_01    id_02  id_03  id_04  id_05  id_06  id_07  id_08  \
0        2987004    0.0  70787.0    NaN    NaN    NaN    NaN    NaN    NaN   
1        2987008   -5.0  98945.0    NaN    NaN    0.0   -5.0    NaN    NaN   

   id_09  ...                id_31  id_32      id_33           id_34  id_35  \
0 

In [17]:
# Merge on TransactionID (standard for IEEE-CIS)
df = train_trans.merge(train_id, on="TransactionID", how="left")

# Target column in IEEE-CIS is usually `isFraud`
assert "isFraud" in df.columns, "❌ Target column 'isFraud' not found"

print_basic_info(df, "merged_df")


--- merged_df ---
Shape: (590540, 434)
Columns: 434
   TransactionID  isFraud  TransactionDT  TransactionAmt ProductCD  card1  \
0        2987000        0          86400            68.5         W  13926   
1        2987001        0          86401            29.0         W   2755   

   card2  card3       card4  card5  ... id_31  id_32  id_33  id_34  id_35  \
0    NaN  150.0    discover  142.0  ...   NaN    NaN    NaN    NaN    NaN   
1  404.0  150.0  mastercard  102.0  ...   NaN    NaN    NaN    NaN    NaN   

  id_36 id_37  id_38  DeviceType  DeviceInfo  
0   NaN   NaN    NaN         NaN         NaN  
1   NaN   NaN    NaN         NaN         NaN  

[2 rows x 434 columns]


## Global Missing Value Analysis (Data Quality Check)

Before modeling, it is important to understand the **overall data quality**.

In this step:
- We analyze missing-value percentages on the **combined dataset**
- This is done **only for diagnostic purposes**
- No features are created and no model decisions are trained here

This helps answer:
- Which columns are mostly empty?
- Which columns may inject noise if blindly imputed?
- Where informed decisions are needed instead of default preprocessing


In [18]:
# Missing value percentage on the FULL dataset (diagnostic only)
missing_pct_all = (
    df.isna()
      .mean()
      .sort_values(ascending=False)
      .rename("missing_ratio")
      .to_frame()
)

missing_pct_all.head(20)


,missing_ratio
id_24,0.991962
id_25,0.991310
id_07,0.991271
id_08,0.991271
id_21,0.991264
id_26,0.991257
id_27,0.991247
id_23,0.991247
id_22,0.991247
dist2,0.936284


In [19]:
missing_pct_all.describe()


,missing_ratio
count,434.000000
mean,0.450744
std,0.365432
min,0.000000
25%,0.002663
50%,0.472935
75%,0.779134
max,0.991962


In [20]:
def bucket_missingness(missing_ratio):
    if missing_ratio < 0.2:
        return "<20%"
    elif missing_ratio < 0.6:
        return "20–60%"
    elif missing_ratio < 0.9:
        return "60–90%"
    else:
        return ">90%"

missing_pct_all["bucket"] = missing_pct_all["missing_ratio"].apply(bucket_missingness)

missing_pct_all["bucket"].value_counts()


,count
bucket,
60–90%,196
<20%,182
20–60%,44
>90%,12


## Final Missing-Value Decisions (Data-Informed)

Based on global missing-value analysis:

| Missing % | Count | Decision |
|----------|------|---------|
| > 90% | 12 | Drop (too sparse, high noise risk) |
| 60–90% | 196 | Retain with caution (missingness may carry signal) |
| 20–60% | 44 | Impute + monitor |
| < 20% | 182 | Standard imputation (safe) |

Key principle:
> Do not apply a single preprocessing rule to all columns.
Decisions must reflect what the data is telling us.


In [21]:
# Identify columns with extreme missingness (>90%)
extreme_missing_cols = missing_pct_all.query("missing_ratio > 0.9").index.tolist()

print("Dropping columns:", extreme_missing_cols)
print("Count:", len(extreme_missing_cols))

# Apply drop BEFORE split (safe because no target info used)
df_reduced = df.drop(columns=extreme_missing_cols)


Dropping columns: ['id_24', 'id_25', 'id_07', 'id_08', 'id_21', 'id_26', 'id_27', 'id_23', 'id_22', 'dist2', 'D7', 'id_18']
Count: 12


In [22]:
X = df_reduced.drop(columns=["isFraud", "TransactionID"])
y = df_reduced["isFraud"].astype(int)

## Leakage Red Flags (Common in Fraud Datasets)

### 1) ID-like columns
- `TransactionID` is an identifier (never use as feature)
- Sometimes other ID columns can act like shortcuts

### 2) Feature engineering before split
- Aggregations computed on full dataset can leak validation info

### 3) Time-based leakage
- If you create features using future time windows, leakage happens

In this notebook:
✅ We will DROP `TransactionID`  
✅ We will SPLIT first  
✅ We will use sklearn Pipelines (safe preprocessing)


In [23]:
# Stratified split (important for imbalance)
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE
)

print("Train shape:", X_train.shape, " | Fraud ratio:", y_train.mean())
print("Val shape  :", X_val.shape,   " | Fraud ratio:", y_val.mean())


Train shape: (472432, 420)  | Fraud ratio: 0.03498916246147594
Val shape  : (118108, 420)  | Fraud ratio: 0.0349933958749619


## Preprocessing Strategy (Leakage-safe)

Fraud datasets have:
- Many missing values
- Mix of numeric + categorical columns
- High-cardinality categories

We will use an sklearn **ColumnTransformer**:
- Numeric: median imputation
- Categorical: most_frequent imputation + OneHotEncoder(handle_unknown="ignore")

This is interview-safe because it:
- Avoids manual leakage
- Keeps the same steps for train and validation
- Makes the workflow reproducible


In [24]:
# Identify column types
numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = X_train.select_dtypes(exclude=[np.number]).columns.tolist()

print("Numeric cols:", len(numeric_cols))
print("Categorical cols:", len(categorical_cols))

# Quick sanity check
print("\nExamples:")
print("Numeric example:", numeric_cols[:5])
print("Categorical example:", categorical_cols[:5])


Numeric cols: 391
Categorical cols: 29

Examples:
Numeric example: ['TransactionDT', 'TransactionAmt', 'card1', 'card2', 'card3']
Categorical example: ['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain']


In [25]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols),
    ],
    remainder="drop"
)

print("✅ Preprocessing pipeline ready")


✅ Preprocessing pipeline ready


## Baseline Model (Important)

Before using AdaBoost, we set a baseline to answer:
- Is the pipeline working?
- How much improvement does AdaBoost add?

Baseline = Logistic Regression (simple, fast, interpretable)
Metric focus:
- ROC-AUC (ranking quality)
- PR-AUC (useful for imbalanced data)


In [26]:
baseline_model = Pipeline(steps=[
    ("preprocess", preprocess),
    ("clf", LogisticRegression(max_iter=1000, n_jobs=-1))
])

baseline_model.fit(X_train, y_train)

val_probs = baseline_model.predict_proba(X_val)[:, 1]
roc = roc_auc_score(y_val, val_probs)
pr  = average_precision_score(y_val, val_probs)

print("✅ Baseline Logistic Regression")
print("ROC-AUC:", round(roc, 5))
print("PR-AUC :", round(pr, 5))


✅ Baseline Logistic Regression
ROC-AUC: 0.69056
PR-AUC : 0.10947


## AdaBoost Model (Resume Model)

Simple explanation:
AdaBoost builds multiple small decision trees and gives more focus to the samples that were misclassified earlier.
This helps in fraud because fraud cases are often "hard examples".

We will start with a safe configuration and then tune lightly.


In [27]:
# A shallow tree is common for AdaBoost (weak learner)
base_tree = DecisionTreeClassifier(max_depth=2, random_state=RANDOM_STATE)

ada_model = Pipeline(steps=[
    ("preprocess", preprocess),
    ("clf", AdaBoostClassifier(
        estimator=base_tree,
        n_estimators=200,
        learning_rate=0.05,
        random_state=RANDOM_STATE
    ))
])

ada_model.fit(X_train, y_train)

val_probs_ada = ada_model.predict_proba(X_val)[:, 1]
roc_ada = roc_auc_score(y_val, val_probs_ada)
pr_ada  = average_precision_score(y_val, val_probs_ada)

print("✅ AdaBoost Evaluation")
print("ROC-AUC:", round(roc_ada, 5))
print("PR-AUC :", round(pr_ada, 5))


✅ AdaBoost Evaluation
ROC-AUC: 0.86135
PR-AUC : 0.41525


## Model Comparison (Leakage-safe)

Baseline Logistic Regression:
- ROC-AUC ≈ 0.69
- PR-AUC  ≈ 0.11

AdaBoost:
- ROC-AUC ≈ 0.86
- PR-AUC  ≈ 0.42

Interpretation:
- Logistic Regression is a linear model → limited for complex fraud patterns
- AdaBoost combines many weak trees and focuses on hard-to-classify cases → better separation and much stronger PR-AUC
- PR-AUC improvement is especially important because fraud is highly imbalanced


In [29]:
import pandas as pd
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score

def eval_threshold(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)

    cm = confusion_matrix(y_true, y_pred)
    cm_df = pd.DataFrame(
        cm,
        index=["Actual: Non-Fraud", "Actual: Fraud"],
        columns=["Predicted: Non-Fraud", "Predicted: Fraud"]
    )

    tn, fp, fn, tp = cm.ravel()
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec  = recall_score(y_true, y_pred, zero_division=0)
    f1   = f1_score(y_true, y_pred, zero_division=0)

    print(f"\nThreshold: {threshold}")
    display(cm_df)
    print(f"TN={tn}, FP={fp}, FN={fn}, TP={tp}")
    print(f"Precision={prec:.4f}, Recall={rec:.4f}, F1={f1:.4f}")

    return {
        "threshold": threshold,
        "tn": tn, "fp": fp, "fn": fn, "tp": tp,
        "precision": prec, "recall": rec, "f1": f1
    }

_ = eval_threshold(y_val, val_probs_ada, threshold=0.5)



Threshold: 0.5


,Predicted: Non-Fraud,Predicted: Fraud
Actual: Non-Fraud,113902,73
Actual: Fraud,3571,562


TN=113902, FP=73, FN=3571, TP=562
Precision=0.8850, Recall=0.1360, F1=0.2357
